In [ ]:

import pandas as pd

import matplotlib.pyplot as plt




This notebook was restructured to improve oversight


In [ ]:
import ollama
#importing ollama and testing if it works with an arbitrary query
response = ollama.chat(model="llama3.1:8b",
                       messages=[{"role": "user", "content": "give me your favorite sentence"}])
print(response["message"]["content"])

In [ ]:

tos_train = pd.read_csv("../data/tos_train.csv")

In [ ]:
#reading the Terms of service training data
tos_train = pd.read_csv("../data/tos_train.csv")



tos_train["original_words"] = tos_train["original_text"].str.split().str.len()
tos_train["summary_words"] = tos_train["reference_summary"].str.split().str.len()
tos_train["sum_/_original"] = tos_train["summary_words"] / tos_train["original_words"]

#The summary/original is to get a clause which is not as compressed - so there is a lower change that information is dropped

candidates = tos_train[tos_train["original_words"] >= 100]
candidates = candidates.sort_values("sum_/_original", ascending=False)
candidates = candidates.head(25)  #get the 25 with the least compression
candidates.to_csv("../results/25_highest_ration.csv", index=False)

#then I hand picked three good examples

selecting the best 3 to use as an example for the LLM to generate the data:

apple administers and operates the www apple com site from its location in cupertino california usa other apple sites may be administered and operated from various locations outside the united states. although the site is accessible worldwide not all features products or services discussed referenced provided or offered through or on the site are available to all persons or in all geographic locations or appropriate or available for use outside the united states. apple reserves the right to limit in its sole discretion the provision and quantity of any feature product or service to any person or geographic area. any offer for any feature product or service made on the site is void where prohibited. if you choose to access the site from outside the united states you do so on your own initiative and you are solely responsible for complying with applicable local laws.

not all apple products or services are available to everyone or in all locations. apple may limit any product or service to any person or area. products and services are not available where prohibited by law. you are responsible for complying with local laws if accessing the website outside of the u s.





you re free to stop using our services at any time. we also reserve the right to suspend or end the services at any time at our discretion and without notice. for example we may suspend or terminate your use of the services if you re not complying with these terms or use the services in a manner that would cause us legal liability disrupt the services or disrupt others use of the services. except for paid accounts we reserve the right to terminate and delete your account if you haven t accessed our services for 12 consecutive months. we ll of course provide you with notice via the email address associated with your account before we do so.

dropbox can end their services at any time without notice. for free accounts they can delete your account if you haven t used dropbox for 12 months. they ll email you before doing so.




billing. you can increase your storage space and add paid features to your account turning your account into a paid account. we ll automatically bill you from the date you convert to a paid account and on each periodic renewal until cancellation. you re responsible for all applicable taxes and we ll charge tax when required to do so. no refunds. you may cancel your dropbox paid account at any time but you won t be issued a refund. downgrades. your paid account will remain in effect until it s cancelled or terminated under these terms. if you don t pay for your paid account on time we reserve the right to suspend it or reduce your storage to free space levels. you re responsible for all taxes and we may charge taxes when required to do so. changes. we may change the fees in effect but will give you advance notice of these changes via a message to the email address associated with your account.

you re responsible to pay any bills you re not entitled to any refunds and if you re late on payments dropbox can suspend or reduce the storage on your account. they can also change their pricing but will email you if they do.

In [ ]:
data_train = pd.read_csv("../data/cuad_train.csv")

SYSTEM_PROMPT = """You are tasked with rewriting legal contract clauses in easier english so that the average person can understand.
Rules:
1: Keep every obligation, permission, prohibition, condition, exception, amount and time period. Also keep who pays for what and any fallback duties (what a party must do if the main duty is not possible).
2: Keep the strength of each duty -> a permission remains a permission
3: dont add elements or information that is not present in the original
4: Use short sentences and simple words -> only surpass 3 sillibals if absolutely necessary
5. Keep placeholders like [REDACTED AMOUNT], [REDACTED], [REDACTED DURATION] etc. exactly as they are
6: write in plain sentences -> DONT use bullet points lists or tables
7: never mention names or companies that are not in the original
final: Reply with the rewritten text only."""

EXAMPLES = [
    (
    "apple administers and operates the www apple com site from its location in cupertino california usa other apple sites may be administered and operated from various locations outside the united states. although the site is accessible worldwide not all features products or services discussed referenced provided or offered through or on the site are available to all persons or in all geographic locations or appropriate or available for use outside the united states. apple reserves the right to limit in its sole discretion the provision and quantity of any feature product or service to any person or geographic area. any offer for any feature product or service made on the site is void where prohibited. if you choose to access the site from outside the united states you do so on your own initiative and you are solely responsible for complying with applicable local laws.",
    "not all apple products or services are available to everyone or in all locations. apple may limit any product or service to any person or area. products and services are not available where prohibited by law. you are responsible for complying with local laws if accessing the website outside of the u s."),

    ("you re free to stop using our services at any time. we also reserve the right to suspend or end the services at any time at our discretion and without notice. for example we may suspend or terminate your use of the services if you re not complying with these terms or use the services in a manner that would cause us legal liability disrupt the services or disrupt others use of the services. except for paid accounts we reserve the right to terminate and delete your account if you haven t accessed our services for 12 consecutive months. we ll of course provide you with notice via the email address associated with your account before we do so.",
     "dropbox can end their services at any time without notice. for free accounts they can delete your account if you haven t used dropbox for 12 months. they ll email you before doing so."),

    ("billing. you can increase your storage space and add paid features to your account turning your account into a paid account. we ll automatically bill you from the date you convert to a paid account and on each periodic renewal until cancellation. you re responsible for all applicable taxes and we ll charge tax when required to do so. no refunds. you may cancel your dropbox paid account at any time but you won t be issued a refund. downgrades. your paid account will remain in effect until it s cancelled or terminated under these terms. if you don t pay for your paid account on time we reserve the right to suspend it or reduce your storage to free space levels. you re responsible for all taxes and we may charge taxes when required to do so. changes. we may change the fees in effect but will give you advance notice of these changes via a message to the email address associated with your account.",

    "you re responsible to pay any bills you re not entitled to any refunds and if you re late on payments dropbox can suspend or reduce the storage on your account. they can also change their pricing but will email you if they do.")
]



In [ ]:
def build_messages(clause):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]  #instructions for model
    for original, simplified in EXAMPLES:
        messages.append({"role": "user", "content": original})  #Example input for model
        messages.append({"role": "assistant", "content": simplified}) #Example output for model
    messages.append({"role": "user", "content": clause}) #attaching clause. Ollama will then continue the pattern given on the new clause
    return messages


def simplify(clause):
    response = ollama.chat(
        model="llama3.1:8b",
        messages=build_messages(clause),  #returns only the output of the model
        options={"temperature": 0, "num_ctx": 4096},  #temperature 0 to have a reproducible output (will pick only the most likely next word and then num_ctx to make sure the context window is big enough
    )
    return response["message"]["content"].strip()

In [ ]:
for clause in data_train["clause_text"].head(3):
    print("ORIGINAL:", clause)
    print("REWRITE: ", simplify(clause))
    print()

#simple test, if model is working as planned

In [ ]:
#DONT RUN!!
#This is the first run of the model
#I found some issues with this approach and slightly changed it below

#outputs = []
#for clause in data_train["clause_text"]:
#    outputs.append(simplify(clause))

#data_train["rewrite"] = outputs
#data_train.to_csv("../data/generated_cuad_raw.csv", index=False)
#print("finished")

In [ ]:
gen = pd.read_csv("../data/generated_cuad_raw.csv")


In [ ]:
from src.faithfulness_filter import is_faithful
from src.readability_filter import readability_drop

forward_scores = []
reverse_scores = []
modal_flips = []
grade_drops = []

for i, row in gen.iterrows():
    temp = is_faithful(row["clause_text"], row["rewrite"])
    forward_scores.append(temp["forward_score"])
    reverse_scores.append(temp["reverse_score"])
    modal_flips.append(temp["modal_flip"])
    grade_drops.append(readability_drop(row["clause_text"], row["rewrite"]))

gen["forward_score"] = forward_scores
gen["reverse_score"] = reverse_scores
gen["modal_flip"] = modal_flips
gen["grade_drop"] = grade_drops
gen.to_csv("../data/generated_scored.csv", index=False)

gen.head(10)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

gen_v1 = pd.read_csv("../data/generated_scored.csv")

fig, axes = plt.subplots(3, 1, figsize=(4, 15))

axes[0].hist(gen_v1["forward_score"], bins=50)
axes[0].axvline(0.16, color="red", linestyle="--")
axes[0].set_title("Forward score")
axes[0].set_xlabel("SummaC score")
axes[0].set_ylabel("number of pairs")

axes[1].hist(gen_v1["reverse_score"], bins=50)
axes[1].axvline(0.4, color="red", linestyle="--")
axes[1].set_title("Reverse score")
axes[1].set_xlabel("SummaC score")
axes[1].set_ylabel("number of pairs")

axes[2].hist(gen_v1["grade_drop"], bins=50)
axes[2].axvline(2.1, color="red", linestyle="--")
axes[2].set_title("Readability: grade drop")
axes[2].set_xlabel("grade levels")
axes[2].set_ylabel("number of pairs")

plt.tight_layout()
plt.savefig("../results/generated_score_distributions_v1.pdf")
plt.show()

In [ ]:
rejected = gen[gen["reverse_score"] < 0.4].sample(20, random_state=1)
#checking whether the filter rejected pairs that should be rejected


for i, row in rejected.iterrows():
    print(round(row["reverse_score"], 2))
    print("ORIGINAL:", row["clause_text"])
    print("REWRITE: ", row["rewrite"])
    print()

Here I realised that the reverse filter is catching to many good examples and therefore I will no longer use it as limitation

In [ ]:
from src.faithfulness_filter import get_modals

#modal flips of v1
gen = pd.read_csv("../data/generated_scored.csv")
flipped = gen[gen["modal_flip"]].copy()


orig_classes = []
rew_classes = []
for i, row in flipped.iterrows():
    orig_classes.append(get_modals(row["clause_text"]))
    rew_classes.append(get_modals(row["rewrite"]))

flipped["orig_classes"] = orig_classes
flipped["rew_classes"] = rew_classes


#one sided = one of the two texts has no modal at all
one_sided = []
for i, row in flipped.iterrows():
    if len(row["orig_classes"]) == 0 or len(row["rew_classes"]) == 0:
        one_sided.append(True)
    else:
        one_sided.append(False)

flipped["one_sided"] = one_sided

print(len(flipped), "flips")
print(flipped["one_sided"].mean(), "of them are one-sided")

In [ ]:
two_sided = flipped[~flipped["one_sided"]].sample(10, random_state=1)

for i, row in two_sided.iterrows():
    print(row["orig_classes"], "->", row["rew_classes"])
    print("ORIGINAL:", row["clause_text"])
    print("REWRITE: ", row["rewrite"])
    print()

After reading through the sample of the rejected pairs, i decided to keep modals only as extra information. It should no longer be a rejection criteria

In [ ]:
candidates_pool = pd.read_csv("../data/cuad_train.csv")
candidates_pool["words"] = candidates_pool["clause_text"].str.split().str.len()

# filtering for clauses containing exceptions and cost rules, as those qualifiers were dropped the most in v1 rewrites

short = candidates_pool[candidates_pool["words"].between(15, 25)]
medium = candidates_pool[
    candidates_pool["words"].between(40, 60)
    & candidates_pool["clause_text"].str.contains("except|provided", case=False)
]
long = candidates_pool[
    candidates_pool["words"].between(90, 120)
    & candidates_pool["clause_text"].str.contains("cost|expense", case=False)
]

# Hand-selected one clause per group as few-shot examples:
# 1500 (short), 1520 (medium, with an exception), 577 (long).
chosen = candidates_pool.loc[[1500, 1520, 577], ["category", "words", "clause_text"]]
print(chosen)

In [ ]:
long

In [ ]:
short

In [ ]:
medium

In [ ]:
#I generated more accurate summaries for these text with an advanced Claude model (with some changes by me) to teach the model to have a higher accuracy

data_train = pd.read_csv("../data/cuad_train.csv")

SYSTEM_PROMPT = """You are tasked with rewriting legal contract clauses in easier english so that the average person can understand.
Rules:
1: Keep every obligation, permission, prohibition, condition, exception, amount and time period. Also keep who pays for what and any fallback duties (what a party must do if the main duty is not possible).
2: Keep the strength of each duty -> a permission remains a permission
3: dont add elements or information that is not present in the original
4: Use short sentences and simple words -> only surpass 3 sillibals if absolutely necessary
5. Keep placeholders like [REDACTED AMOUNT], [REDACTED], [REDACTED DURATION] etc. exactly as they are
6: write in plain sentences -> DONT use bullet points lists or tables
7: never mention names or companies that are not in the original
final: Reply with the rewritten text only."""

EXAMPLES = [
    (
        "Further, during the Non-Competition Period, Rubicon shall not solicit any employee of ISA or any employee of any ISA Client.",
        "During the Non-Competition Period, Rubicon must not try to hire anyone who works for ISA or for any of ISA's clients.",
    ),
    (
        "Such rights to Investor Improvements shall be solely for use by the Company and its Affiliates and shall not be transferable to any Third Party, except in connection with a merger, consolidation, or the sale or transfer of substantially all of Company's assets associated with performance under this IP Agreement.",
        "These rights to Investor Improvements must be used only by the Company and its Affiliates. They must not be passed on to any Third Party, except as part of a merger, a consolidation, or a sale or transfer of almost all of the Company's assets used to carry out this IP Agreement.",
    ),
    (
        "At the Supplier's option, on termination of this agreement: (a) the Supplier may buy from the Distributor all or any stocks of Products for the current market value for those Products. The Distributor must deliver such Products to the Supplier within 14 days of receiving the Supplier's notice, and the Supplier must pay for the Products in full within 30 days of their delivery. The Supplier shall be responsible for the costs of packaging, insurance and carriage of the Products; or (b) the Distributor may dispose of the balance of the Products in its possession and account to the Supplier for the Price for those Products;",
        "When this agreement ends, the Supplier chooses one of two options. First, the Supplier may buy some or all of the Distributor's stock of Products at their current market value. In that case, the Distributor must deliver the Products within 14 days of getting the Supplier's notice, and the Supplier must pay for them in full within 30 days of delivery. The Supplier must also pay for packing, insurance and transport. Second, the Distributor may sell or otherwise get rid of the rest of the Products it has, and must pay the Supplier the Price for them.",
    ),
]

to_generate = data_train.drop([1500, 1520, 577]).copy()
print(len(data_train), len(to_generate))



In [ ]:
#DONT RUN!! takes hours and overwrites the saved v2 outputs, which the rest of the notebook loads

outputs = []
for clause in to_generate["clause_text"]:
    outputs.append(simplify(clause))

to_generate["rewrite"] = outputs
to_generate.to_csv("../data/generated_raw_v2(selfmade_summaries).csv", index=False)
print("finished")

In [ ]:
import pandas as pd
from src.faithfulness_filter import model_CZS, modal_flip_detected  #using them directly to not have reverse score as a limitation
from src.readability_filter import readability_drop

gen2 = pd.read_csv("../data/generated_raw_v2(selfmade_summaries).csv")

forward_scores = []
modal_flips = []
grade_drops = []

for i, row in gen2.iterrows():
    forward_scores.append(model_CZS.score([row["clause_text"]], [row["rewrite"]])["scores"][0])
    modal_flips.append(modal_flip_detected(row["clause_text"], row["rewrite"]))
    grade_drops.append(readability_drop(row["clause_text"], row["rewrite"]))

gen2["forward_score"] = forward_scores
gen2["modal_flip"] = modal_flips
gen2["grade_drop"] = grade_drops
gen2.to_csv("../data/generated_scored_v2.csv", index=False)

In [ ]:
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(4, 10))

axes[0].hist(gen2["forward_score"], bins=50)
axes[0].axvline(0.16, color="red", linestyle="--")
axes[0].set_title("v2 Forward score")
axes[0].set_xlabel("SummaC score")
axes[0].set_ylabel("number of pairs")

axes[1].hist(gen2["grade_drop"], bins=50)
axes[1].axvline(2.1, color="red", linestyle="--")
axes[1].set_title("v2 Readability: grade drop")
axes[1].set_xlabel("grade levels")
axes[1].set_ylabel("number of pairs")

plt.tight_layout()
plt.savefig("../results/generated_score_distributions_v2.pdf")
plt.show()

In [ ]:
gen2 = pd.read_csv("../data/generated_scored_v2.csv")

forward_ok = gen2["forward_score"] >= 0.16
no_flip = ~gen2["modal_flip"]
readable = gen2["grade_drop"] >= 2.1
passes = forward_ok & readable

print("forward:    ", forward_ok.mean())
print("no flip:    ", no_flip.mean())
print("readability:", readable.mean())
print("all:        ", passes.mean(), "->", passes.sum(), "pairs")

In [ ]:
v1 = pd.read_csv("../data/generated_scored.csv")
v2 = pd.read_csv("../data/generated_scored_v2.csv")


#v1
forward_ok = v1["forward_score"] >= 0.16
no_flip = ~v1["modal_flip"]
readable = v1["grade_drop"] >= 2.1
passes = forward_ok & readable

print("v1")
print("forward:  ", forward_ok.mean())
print("no flip:  ", no_flip.mean())
print("readability:", readable.mean())
print("all:  ", passes.mean(), "->", passes.sum(), "pairs")


#v2
forward_ok = v2["forward_score"] >= 0.16
no_flip = ~v2["modal_flip"]
readable = v2["grade_drop"] >= 2.1
passes = forward_ok & readable

print("v2")
print("forward: ", forward_ok.mean())
print("no flip: ", no_flip.mean())
print("readability:", readable.mean())
print("all: ", passes.mean(), "->", passes.sum(), "pairs")

v2 keeps fewer pairs (1,383 vs. 1,443). It's pass rate is 5% higher, but as it is more detailed, the rewrites fail the readability threshold more often. I chose v2 because after some manual inspection I found fewer qualifiers and invented details, which the automatic filter cannot detect.

In [ ]:
for t in [0.16, 0.3, 0.5, 0.7]:
    kept = (gen2["forward_score"] >= t) & readable
    print(t, kept.sum())

In [ ]:
final_train = gen2[(gen2["forward_score"] >= 0.16) & readable]

final_train.to_csv("../data/stage1_train.csv", index=False)
